# Experiment 01 — capacity-matched classical control under DQN

Colab driver. This notebook is deliberately **thin**: it clones the repository,
prepares the environment and calls the package. All logic lives in `src/qrl_dissection/`
and `experiments/`, so what you run here is what a reviewer reads on GitHub.

Read `docs/EXPERIMENT-01.md` for the design and `docs/CORRECTIONS.md` for the
corrections registry.

**Workflow.** Code in GitHub, results in Drive. The repository is cloned fresh on
each session; run artefacts are written to Drive and never committed. To change
code, push from your machine and re-run section 1 — do not edit files inside the
Colab checkout, they vanish with the session.

---
## 1. Repository and environment

Set `REPO_URL` to your fork. Public repository → the clone needs no token.

In [ ]:
from google.colab import userdata

GITHUB_USER = "RogerMas99"
REPO_NAME = "qrl-dissection"
BRANCH = "main"

try:
    GH_TOKEN = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{GH_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if GH_TOKEN else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import sys, subprocess, pathlib, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp01")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd() / "results" / "exp01"
    CODE    = pathlib.Path.cwd()
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if CODE.exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "--quiet"], check=False)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE)], check=True)

rev = subprocess.run(["git", "-C", str(CODE), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print("code   :", CODE, "@", rev)
print("results:", RESULTS)

In [ ]:
# Pins come from the repository, not from this notebook -> one source of truth.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
# SimplyQRL is pinned inside requirements.txt (audited revision), so the line
# above already installed it at the correct commit. Do NOT install it separately
# here: an unpinned install would silently override the pin.

# [FIX-04] jax is in neither upstream lock; Colab preinstalls it and PennyLane
# imports it opportunistically. jax >= 0.6.0 removed jax.core.Primitive.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

# --- Reiniciar el entorno SOLO si es necesario --------------------------------
# Un modulo ya importado no se reemplaza en caliente. Esto solo muerde si Colab
# trajo cargada de fabrica una autoray incompatible (o jax) ANTES de este install.
# En un runtime limpio, ejecutado en orden, no hace falta reiniciar.
import importlib, sys as _sys

_need_restart = False
if "autoray.autoray" in _sys.modules:
    import autoray.autoray as _aa
    _need_restart = not hasattr(_aa, "NumpyMimic")
if "jax" in _sys.modules:
    _need_restart = True

if _need_restart:
    print("Versiones incompatibles ya cargadas en memoria -> reiniciando el entorno.")
    print("Colab relanzara el runtime solo. Cuando vuelva, continua desde la")
    print("celda 'After restarting, run from here' (NO re-ejecutes las de arriba).")
    import os as _os
    _os.kill(_os.getpid(), 9)   # Colab detecta la caida y relanza el runtime
else:
    print("=" * 66)
    print("Entorno limpio: NO hace falta reiniciar. Continua con la siguiente celda.")
    print("(Si en otra sesion aparece el aviso de reinicio, dejalo actuar: se")
    print(" reinicia solo y sigues desde la celda de despues del reinicio.)")
    print("=" * 66)


### Después del reinicio, ejecuta desde aquí

Si la celda anterior dijo *«NO hace falta reiniciar»*, esta celda y la de instalación redefinen lo necesario de todos modos, así que puedes seguir sin más. Si hubo reinicio, empieza aquí: las variables de la celda 1 se perdieron y se reconstruyen abajo.

In [ ]:
import sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/exp01")
    CODE    = pathlib.Path("/content/qrl-dissection")
else:
    RESULTS = pathlib.Path.cwd() / "results" / "exp01"; CODE = pathlib.Path.cwd()

print(subprocess.run([sys.executable, str(CODE / "scripts" / "verify_env.py")],
                     capture_output=True, text=True).stdout)

In [ ]:
# Fast checks: probe logic, capacity matching, both upstream patches. ~10 s.
print(subprocess.run([sys.executable, "-m", "pytest", "-q", str(CODE / "tests")],
                     capture_output=True, text=True).stdout[-3000:])

---
## 2. Capacity ladder — trains nothing

The gate before spending compute. Confirm that `paper_linear` really builds
`Linear(12, 2)` with ~26 parameters, and read off the PQC budget that
`matched_classical` is sized against. If a count looks wrong, stop here: the arm
is not what we think it is and the grid would be worthless.

In [ ]:
r = subprocess.run([sys.executable, str(CODE / "experiments" / "exp01_dqn_cartpole_capacity.py"),
                    "--ladder-only"], capture_output=True, text=True)
print(r.stdout or r.stderr)

---
## 3. Throughput probe

One short run before committing hours. The estimate is optimistic: it runs below
`learning_starts=10000`, so the per-step cost rises once updates begin.

In [ ]:
import time
sys.path.insert(0, str(CODE / "src"))
import qrl_dissection
from qrl_dissection.dqn import RunSpec, run_arm

t0 = time.time()
_ = run_arm(RunSpec(arm="paper_linear", seed=99, fix_autoreset=True,
                    total_timesteps=2000, tag="probe"),
            outdir=RESULTS / "_probe")
dt = time.time() - t0
print(f"{2000/dt:.0f} steps/s  ->  60k steps ~ {60_000/dt/60:.1f} min per classical run")
print(f"12-cell classical grid ~ {12 * 60_000/dt/60:.0f} min (optimistic)")

---
## 4. The grid

3 arms × FIX-01 {off, on} × 3 seeds × 60k steps.

Resumable: each finished cell writes a manifest and is skipped on re-run, so a
disconnect costs you one cell, not the grid. Upstream flushes its episode CSV
once per episode, so even the interrupted cell leaves a usable partial curve.

In [ ]:
from qrl_dissection.dqn import GreedyEvalConfig, run_grid

ARMS  = ["paper_linear", "matched_classical", "oversized_mlp"]
SEEDS = [1, 2, 3]
STEPS = 60_000

specs = [RunSpec(arm=arm, seed=seed, fix_autoreset=fix, total_timesteps=STEPS,
                 dqn_kwargs=dict(batch_size=128, buffer_size=10_000, train_frequency=10))
         for arm in ARMS for fix in (False, True) for seed in SEEDS]

results = run_grid(specs, RESULTS, eval_cfg=GreedyEvalConfig(every_steps=10_000))
print(f"\n{sum('error' not in r for r in results)}/{len(results)} cells ok")

---
## 5. Results

`best_ma50` is the headline metric, not `tail_mean`. DQN on CartPole decays from
its peak even when healthy, so a tail statistic reports failure on runs that
reached 200–400 mid-training.

In [ ]:
from qrl_dissection import analysis

df = analysis.to_dataframe(RESULTS)
display(df.sort_values(["arm", "fix01", "seed"]))
print()
print(analysis.arm_comparison(RESULTS, metric="best_ma50"))

for arm in df.arm.unique():
    off = df[(df.arm == arm) & (~df.fix01)].best_ma50
    on  = df[(df.arm == arm) & (df.fix01)].best_ma50
    print(f"{arm:<20} FIX-01 off {off.mean():7.1f} +/- {off.std():5.1f} | "
          f"on {on.mean():7.1f} +/- {on.std():5.1f} | delta {on.mean()-off.mean():+7.1f}")

In [ ]:
fig = analysis.plot_arms(RESULTS, savepath=str(RESULTS / "exp01_arms.png"))
fig

---
## 6. Recording the result

Paste the table from section 5 into `docs/RESULTS-LOG.md` under *Experiment 01*,
together with the git revision printed in section 1 and the `verify_env.py`
output. Commit that file from your machine.

The point of the log is that the repository stays self-contained: someone
cloning it later can follow what was run, against which code, in which
environment — without access to your Drive.

**How to read the outcome** (full version in `docs/EXPERIMENT-01.md` §6):

- `matched_classical` learns → `paper_linear` died of capacity, not of the
  absence of a circuit. The paper's parity design does not transfer off-policy,
  and you gain a second live configuration in which FIX-01 is measurable.
- `matched_classical` dies → at equal parameter budget the classical block does
  not reach where the PQC does. A positive result for the circuit, and stronger
  than anything the paper reports.
- Mixed across seeds → report it as a variance result; seed dispersion in this
  regime is already known to be a factor of two.